# StyleTTS2 Humanization Notebook

This notebook processes raw TTS audio files using StyleTTS2 for prosody transfer and natural speech enhancement.

## Method: Prosody Transfer with StyleTTS2
- Uses StyleTTS2 for prosody modeling and transfer
- Applies natural prosodic patterns to TTS output
- High-quality prosody-aware re-synthesis
- Can use reference human audio for prosody patterns (optional)

## Input/Output:
- **Input**: `outputs/raw/{batch_run_id}/{speaker_id}/{prompt_id}.wav`
- **Output**: `outputs/styletts2/{batch_run_id}/{speaker_id}/{prompt_id}.wav`


In [ ]:
# Cell 1: Environment Setup and Path Detection

import os
from pathlib import Path

# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("STYLETTS2 HUMANIZATION - Environment Setup")
print("=" * 80)

if IN_COLAB:
    print("Detected: Google Colab environment")
    print("\nMounting Google Drive...")
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Libri_Vibevoice")
else:
    print("Detected: Local environment")
    BASE_DIR = Path.cwd()
    if "Libri_Vibevoice" not in str(BASE_DIR):
        BASE_DIR = Path(r"G:\My Drive\Libri_Vibevoice")
    print(f"Using BASE_DIR: {BASE_DIR}")

# Set up paths
INPUT_DIR = BASE_DIR / "outputs" / "raw"
OUTPUT_DIR = BASE_DIR / "outputs" / "styletts2"

print(f"\nBASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

# Verify input directory exists
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"Input directory not found at: {INPUT_DIR}")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n✓ Setup complete. Input directory found: {INPUT_DIR}")
print(f"✓ Humanized outputs will be saved to: {OUTPUT_DIR}")


In [ ]:
# Cell 2: Install Dependencies

import subprocess
import sys

print("=" * 80)
print("Installing Required Packages")
print("=" * 80)

packages = [
    "torch>=2.0.0",
    "transformers>=4.30.0",
    "librosa>=0.10.0",
    "soundfile>=0.12.0",
    "numpy>=1.24.0",
    "scipy>=1.10.0",
    "tqdm>=4.65.0",
]

# Install StyleTTS2 (clone and install from GitHub)
if IN_COLAB:
    print("Cloning StyleTTS2 repository...")
    subprocess.run(["git", "clone", "https://github.com/yl4579/StyleTTS2.git", "/content/StyleTTS2"], check=False)
    sys.path.insert(0, "/content/StyleTTS2")
else:
    print("Note: For local installation, clone StyleTTS2 manually:")
    print("  git clone https://github.com/yl4579/StyleTTS2.git")
    print("  cd StyleTTS2")
    print("  pip install -r requirements.txt")

for package in packages:
    try:
        print(f"Installing {package}...")
        if IN_COLAB:
            subprocess.run([sys.executable, "-m", "pip", "install", package, "--quiet"], check=True)
        else:
            subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"⚠ Warning: Failed to install {package}: {e}")

print("\n✓ Dependencies installation complete")


In [ ]:
# Cell 3: Load StyleTTS2 Model

import torch

print("=" * 80)
print("Loading StyleTTS2 Model")
print("=" * 80)

# Device selection
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple Silicon GPU")
else:
    DEVICE = "cpu"
    print("Using CPU")

# Note: StyleTTS2 requires specific model loading
# This is a simplified version - adjust based on actual StyleTTS2 API
print("\nNote: StyleTTS2 model loading requires:")
print("  1. Downloading pre-trained models")
print("  2. Initializing the StyleTTS2 pipeline")
print("  3. Loading prosody extractor")

# Placeholder for StyleTTS2 initialization
# In practice, you would:
# from StyleTTS2 import StyleTTS2
# model = StyleTTS2(model_path="path/to/model")
# model.to(DEVICE)

print("⚠ StyleTTS2 model loading needs to be implemented based on actual API")
print("Please refer to StyleTTS2 documentation for proper initialization")


In [ ]:
# Cell 4: StyleTTS2 Processing Functions

import librosa
import soundfile as sf
import numpy as np

def extract_prosody(audio, sr=24000):
    """
    Extract prosody features from audio (optional - can use reference human audio).
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
    
    Returns:
        Prosody features
    """
    # Extract prosodic features (F0, energy, duration patterns)
    # This is a simplified version - StyleTTS2 has specific prosody extraction
    f0 = librosa.yin(audio, fmin=50, fmax=400, sr=sr)
    energy = np.abs(audio)
    
    return {
        'f0': f0,
        'energy': energy,
    }


def apply_prosody_transfer(audio, sr=24000, reference_prosody=None):
    """
    Apply prosody transfer using StyleTTS2.
    
    Args:
        audio: Audio signal as numpy array
        sr: Sample rate
        reference_prosody: Optional prosody features from reference audio
    
    Returns:
        Processed audio with natural prosody
    """
    # Extract current prosody
    current_prosody = extract_prosody(audio, sr)
    
    # Apply StyleTTS2 prosody transfer
    # Note: This is a placeholder - actual implementation depends on StyleTTS2 API
    # In practice:
    # processed_audio = styletts2_model.transfer_prosody(
    #     audio, 
    #     target_prosody=reference_prosody or current_prosody
    # )
    
    # For now, return original audio (to be replaced with actual StyleTTS2 processing)
    processed_audio = audio
    
    return processed_audio, sr


def process_audio_file(input_path, output_path):
    """
    Process a single audio file through StyleTTS2.
    
    Args:
        input_path: Path to input audio file
        output_path: Path to save processed audio
    
    Returns:
        True if successful, False otherwise
    """
    try:
        # Load audio
        audio, sr = librosa.load(str(input_path), sr=None)
        
        # Apply prosody transfer
        processed_audio, processed_sr = apply_prosody_transfer(audio, sr)
        
        # Ensure output directory exists
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save processed audio
        sf.write(str(output_path), processed_audio, processed_sr)
        
        return True
    except Exception as e:
        print(f"  ✗ Error processing {input_path}: {e}")
        import traceback
        traceback.print_exc()
        return False

print("✓ StyleTTS2 processing functions defined")
print("⚠ Note: Actual StyleTTS2 API calls need to be implemented")


In [ ]:
# Cell 6: Test Single File (Optional)

print("=" * 80)
print("Test Single File")
print("=" * 80)

# Select a test file (modify these as needed)
TEST_BATCH_RUN_ID = "25094444"  # Change to test different batch run
TEST_SPEAKER_ID = "196"  # Change to test different speaker
TEST_FILENAME = "multi_turn_base_19.wav"  # Change to test different file

test_input_path = INPUT_DIR / TEST_BATCH_RUN_ID / TEST_SPEAKER_ID / TEST_FILENAME
test_output_path = OUTPUT_DIR / TEST_BATCH_RUN_ID / TEST_SPEAKER_ID / TEST_FILENAME.replace('.wav', '_test.wav')

if not test_input_path.exists():
    print(f"⚠ Test file not found: {test_input_path}")
    print("Please update TEST_BATCH_RUN_ID, TEST_SPEAKER_ID, or TEST_FILENAME above.")
else:
    print(f"Testing with: {test_input_path}")
    
    # Process the test file
    if process_audio_file(test_input_path, test_output_path):
        print(f"✓ Test file processed successfully")
        print(f"  Original: {test_input_path}")
        print(f"  Humanized: {test_output_path}")
        
        # Play audio if in Colab
        if IN_COLAB:
            from IPython.display import Audio, display
            print("\nPlaying original audio:")
            display(Audio(str(test_input_path)))
            print("\nPlaying humanized audio:")
            display(Audio(str(test_output_path)))
        else:
            print("\nTo listen to the audio, open the files:")
            print(f"  Original: {test_input_path}")
            print(f"  Humanized: {test_output_path}")
    else:
        print("✗ Failed to process test file")



In [ ]:
# Cell 5: Batch Processing

from pathlib import Path
from tqdm import tqdm

print("=" * 80)
print("Batch Processing - Humanizing Audio Files with StyleTTS2")
print("=" * 80)

BATCH_RUNS_TO_PROCESS = 'all'

files_to_process = []

if BATCH_RUNS_TO_PROCESS == 'all':
    if INPUT_DIR.exists():
        batch_runs = [d for d in INPUT_DIR.iterdir() if d.is_dir()]
    else:
        batch_runs = []
else:
    batch_runs = [INPUT_DIR / run_id for run_id in BATCH_RUNS_TO_PROCESS if (INPUT_DIR / run_id).exists()]

print(f"Found {len(batch_runs)} batch run(s) to process")

for batch_run_dir in batch_runs:
    batch_run_id = batch_run_dir.name
    
    for speaker_dir in batch_run_dir.iterdir():
        if not speaker_dir.is_dir():
            continue
        
        speaker_id = speaker_dir.name
        
        for wav_file in speaker_dir.glob("*.wav"):
            relative_path = wav_file.relative_to(INPUT_DIR)
            output_path = OUTPUT_DIR / relative_path
            files_to_process.append((wav_file, output_path))

print(f"\nFound {len(files_to_process)} WAV files to process")

if len(files_to_process) == 0:
    print("⚠ No files found to process.")
else:
    successful = 0
    failed = 0
    
    for input_path, output_path in tqdm(files_to_process, desc="Processing with StyleTTS2"):
        if process_audio_file(input_path, output_path):
            successful += 1
        else:
            failed += 1
    
    print("\n" + "=" * 80)
    print("Processing Complete")
    print("=" * 80)
    print(f"✓ Successfully processed: {successful} files")
    if failed > 0:
        print(f"✗ Failed: {failed} files")
    print(f"\nHumanized files saved to: {OUTPUT_DIR}")
